In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté
# On fusionne les deux types de données du Bronze (BATCH et STREAMING)

# lecture Silver
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

df_silver = spark.table("transactions_silver")

# Enrichissement Time en miutes / heures /jour
df_gold_base = (
    df_silver
    .withColumn("minute", (F.col("time") / 60).cast("int"))
    .withColumn("hour", (F.col("time") / 3600).cast("int"))
    .withColumn("day", (F.col("time") / 86400).cast("int"))
)

is_fraud_cond = F.col("is_fraud") == 1

df_kpi = (
    df_silver
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("amount").alias("total_amount"),
        F.sum(F.col("is_fraud")).alias("nb_fraud"),
        F.sum(F.when(is_fraud_cond, F.col("amount")).otherwise(0)).alias("fraud_amount"),  # noqa: E501
        F.sum(F.when(~is_fraud_cond, F.col("amount")).otherwise(0)).alias("legit_amount"),  # noqa: E501
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

# Fraude par minute
df_fraud_by_minute = (
    df_gold_base
    .groupBy("minute")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

# Fraude par heure
df_fraud_by_hour = (
    df_gold_base
    .groupBy("hour")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

# Fraude par jour
df_fraud_by_day = (
    df_gold_base
    .groupBy("day")
    .agg(
        F.count("*").alias("nb_transactions"),
        F.sum("is_fraud").alias("nb_fraud"),
        F.sum("amount").alias("total_amount")
    )
    .withColumn("fraud_rate", F.col("nb_fraud") / F.col("nb_transactions"))
)

# Écriture dans Unity Catalog
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA gold")

df_kpi.write.format("delta").mode("overwrite").saveAsTable("fraud_kpi_gold")  # noqa: E501
df_fraud_by_minute.write.format("delta").mode("overwrite").saveAsTable("fraud_by_minute_gold")  # noqa: E501
df_fraud_by_hour.write.format("delta").mode("overwrite").saveAsTable("fraud_by_hour_gold")  # noqa: E501
df_fraud_by_day.write.format("delta").mode("overwrite").saveAsTable("fraud_by_day_gold")  # noqa: E501